In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp('2024-01-23 23:03:29', tz='UTC')
window_end   = pd.Timestamp('2024-01-23 23:33:29', tz='UTC')

# Helper to compute summaries for metric/trace/log style tables
def summarize_timeseries(df, id_col, name_col, value_col, window_start, window_end, top_n=20):
    # mark in-window
    in_window_mask = (df['timestamp'] >= window_start) & (df['timestamp'] <= window_end)
    df['_in_window'] = in_window_mask.astype(int)
    # group aggregates (global on full series)
    grp = df.groupby([id_col, name_col], dropna=False)
    agg = grp.agg(
        total_count = ('timestamp', 'size'),
        count_within_incident_window = ('_in_window', 'sum'),
        min = (value_col, 'min'),
        max = (value_col, 'max'),
        mean = (value_col, 'mean'),
        earliest_ts = ('timestamp', 'min'),
        latest_ts = ('timestamp', 'max'),
    ).reset_index()
    # global P95 per series (calculated on full series before any filtering)
    p95 = grp[value_col].quantile(0.95).reset_index().rename(columns={value_col: 'p95'})
    summary = agg.merge(p95, on=[id_col, name_col], how='left')
    # tidy columns and sorting
    summary = summary.sort_values(by='count_within_incident_window', ascending=False).reset_index(drop=True)
    # keep top N and select columns to display compactly
    display_cols = [id_col, name_col, 'total_count', 'count_within_incident_window', 'min', 'max', 'mean', 'p95', 'earliest_ts', 'latest_ts']
    return summary, summary[display_cols].head(top_n)

# Metric summary
metric_full_summary, metric_summary_top20 = summarize_timeseries(
    metric_df, 'cmdb_id', 'kpi_name', 'value', window_start, window_end, top_n=20
)

# Trace summary
trace_full_summary, trace_summary_top20 = summarize_timeseries(
    trace_df, 'cmdb_id', 'trace_name', 'value', window_start, window_end, top_n=20
)

# Log summary
log_full_summary, log_summary_top20 = summarize_timeseries(
    log_df, 'cmdb_id', 'log_name', 'value', window_start, window_end, top_n=20
)

# Error logs summary: counts and up to 10 sample rows within window
error_df['_in_window'] = ((error_df['timestamp'] >= window_start) & (error_df['timestamp'] <= window_end)).astype(int)
error_counts = {
    'total_error_rows': int(len(error_df)),
    'error_rows_within_window': int(error_df['_in_window'].sum())
}
error_samples_in_window = error_df[error_df['_in_window'] == 1][['timestamp', 'cmdb_id', 'message']].head(10)

# Display compact summaries (as variables for IPython)
metric_summary_top20, trace_summary_top20, log_summary_top20, error_counts, error_samples_in_window

```
Out[1]:
```
summary = (
    "Summary of telemetry for incident window 2024-01-23 23:03:29–23:33:29 UTC:\n\n"
    "1) Error logs: No error log rows were found (total_error_rows = 0; error_rows_within_window = 0). "
    "There are no sample error messages inside the window.\n\n"
    "2) Metrics & Traces: The metric/trace/log time series that appear in the top results all have their full series "
    "falling inside the incident window (most series show total_count = 25 and count_within_incident_window = 25), "
    "with timestamps ranging roughly 2024-01-23 23:06:00 to 2024-01-23 23:30:00 UTC.\n\n"
    "3) Top components by telemetry activity (from the aggregated top rows):\n"
    "- ts-travel2-service: many trace series (calls to ts-order, ts-route, ts-seat, admin-travel) — strong trace activity to investigate.\n"
    "- ts-admin-basic-info-service: multiple metric KPIs present (cpu, diskio, mem, latency, socket, workload).\n"
    "- ts-basic-service and ts-order-service: very high log row counts (high request/row volumes).\n"
    "- ts-voucher-service and several other services appear in metric top rows as well.\n\n"
    "4) Conclusion & recommended next steps:\n"
    "- There are no error log messages to point to a clear failure reason.\n"
    "- Focus investigation on ts-travel2-service traces (latency/p95 and call counts) and on resource KPIs for ts-admin-basic-info-service. "
    "Also review high log volumes in ts-basic-service / ts-order-service for signs of overload.\n"
    "- Collect more detailed span-level traces, application logs (non-aggregated), and host-level resource metrics during the window to determine root cause."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id    kpi_name  total_count  count_within_incident_window           min           max          mean           p95               earliest_ts                 latest_ts
0            ts-voucher-service      socket           25                            25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
1   ts-admin-basic-info-service         cpu           25                            25  3.097646e-01  7.858118e-01  4.833814e-01  6.502908e-01 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
2   ts-admin-basic-info-service      diskio           25                            25  4.271203e+04  6.065199e+04  5.093948e+04  5.834922e+04 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
3   ts-admin-basic-info-service       error           25                            25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
4   ts-admin-basic-info-service  latency-50           25                            25  1.705658e-02  1.774298e-02  1.742541e-02  1.762936e-02 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
5   ts-admin-basic-info-service  latency-90           25                            25  2.341132e-02  2.393737e-02  2.352079e-02  2.377564e-02 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
6   ts-admin-basic-info-service         mem           25                            25  2.290662e+08  2.316217e+08  2.306553e+08  2.315979e+08 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
7   ts-admin-basic-info-service      socket           25                            25  6.766667e+00  8.333333e+00  7.440043e+00  8.000000e+00 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
8   ts-admin-basic-info-service    workload           25                            25  5.410667e-01  7.677333e-01  6.479988e-01  7.475300e-01 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
9        ts-admin-order-service         cpu           25                            25  8.792244e-02  2.775303e-01  1.069908e-01  1.360046e-01 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
10       ts-admin-order-service      diskio           25                            25  2.927493e+04  5.404723e+04  4.283130e+04  5.073538e+04 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
11       ts-admin-order-service         mem           25                            25  1.963613e+08  1.965515e+08  1.964906e+08  1.965498e+08 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
12       ts-admin-order-service      socket           25                            25  3.000000e+00  3.750000e+00  3.075333e+00  3.260000e+00 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
13       ts-admin-route-service         cpu           25                            25  8.851819e-02  3.245125e-01  1.183223e-01  1.445524e-01 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
14       ts-admin-route-service      diskio           25                            25  3.770223e+04  6.203901e+04  5.120317e+04  6.070326e+04 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
15             ts-travel2-mongo      socket           25                            25  8.000000e+00  8.000000e+00  8.000000e+00  8.000000e+00 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
16           ts-travel2-service         cpu           25                            25  3.672357e+00  5.444397e+00  4.264484e+00  4.677385e+00 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
17           ts-travel2-service      diskio           25                            25  4.161780e+04  6.452578e+04  5.219854e+04  6.157695e+04 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
18           ts-travel2-service       error           25                            25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
19           ts-travel2-service  latency-50           25                            25  1.327421e-02  1.621115e-02  1.504158e-02  1.599406e-02 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00,                cmdb_id                                        trace_name  total_count  count_within_incident_window          min          max         mean          p95               earliest_ts                 latest_ts
0   ts-travel2-service     trace.to_ts-order-other-service.duration_mean           25                            25     0.005105     0.007578     0.006416     0.007475 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
1   ts-travel2-service      trace.to_ts-order-other-service.duration_p95           25                            25     0.006500     0.014511     0.010256     0.014122 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
2   ts-travel2-service        trace.to_ts-order-other-service.error_rate           25                            25     0.000000     0.000000     0.000000     0.000000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
3   ts-travel2-service         trace.to_ts-order-other-service.row_count           25                            25    38.000000    88.000000    75.400000    84.000000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
4   ts-travel2-service           trace.to_ts-route-service.duration_mean           25                            25     0.003207     0.003447     0.003324     0.003435 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
5   ts-travel2-service            trace.to_ts-route-service.duration_p95           25                            25     0.003827     0.004506     0.004082     0.004347 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
6   ts-travel2-service              trace.to_ts-route-service.error_rate           25                            25     0.000000     0.000000     0.000000     0.000000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
7   ts-travel2-service               trace.to_ts-route-service.row_count           25                            25   296.000000   651.000000   566.440000   622.400000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
8   ts-travel2-service          trace.from_ts-seat-service.duration_mean           25                            25     0.009201     0.009890     0.009555     0.009758 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
9   ts-travel2-service           trace.from_ts-seat-service.duration_p95           25                            25     0.011356     0.014647     0.012551     0.014245 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
10  ts-travel2-service             trace.from_ts-seat-service.error_rate           25                            25     0.000000     0.000000     0.000000     0.000000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
11  ts-travel2-service              trace.from_ts-seat-service.row_count           25                            25   152.000000   370.000000   307.160000   345.200000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
12  ts-travel2-service       trace.from_ts-travel2-service.duration_mean           25                            25     0.009341     0.009771     0.009584     0.009742 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
13  ts-travel2-service        trace.from_ts-travel2-service.duration_p95           25                            25     0.043845     0.046199     0.045131     0.045725 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
14  ts-travel2-service          trace.from_ts-travel2-service.error_rate           25                            25     0.000000     0.000000     0.000000     0.000000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
15  ts-travel2-service           trace.from_ts-travel2-service.row_count           25                            25  1272.000000  2944.000000  2500.960000  2784.800000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
16  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean           25                            25     0.062271     0.069078     0.066190     0.068691 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
17  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95           25                            25     0.063888     0.082063     0.073642     0.081218 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
18  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate           25                            25     0.000000     0.000000     0.000000     0.000000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
19  ts-travel2-service      trace.from_ts-admin-travel-service.row_count           25                            25     5.000000    12.000000     9.320000    11.800000 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00,                         cmdb_id         log_name  total_count  count_within_incident_window   min   max     mean     p95               earliest_ts                 latest_ts
0   ts-admin-basic-info-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
1   ts-admin-basic-info-service    log.row_count           25                            25    14    34    27.84    33.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
2       ts-admin-travel-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
3       ts-admin-travel-service    log.row_count           25                            25    15    36    27.96    35.4 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
4          ts-assurance-service    log.row_count           25                            25     4    19     9.80    15.8 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
5          ts-assurance-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
6              ts-basic-service    log.row_count           25                            25  1204  2784  2417.52  2751.2 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
7              ts-basic-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
8               ts-auth-service    log.row_count           25                            25   174   352   316.80   350.8 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
9               ts-auth-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
10          ts-contacts-service    log.row_count           25                            25    18    66    40.08    64.8 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
11          ts-contacts-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
12           ts-consign-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
13           ts-consign-service    log.row_count           25                            25    21    77    43.68    68.6 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
14             ts-order-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
15             ts-order-service    log.row_count           25                            25   182   524   381.40   481.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
16          ts-preserve-service    log.row_count           25                            25    26   195   108.76   185.2 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
17          ts-preserve-service  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
18         ts-order-other-mongo  log.error_count           25                            25     0     0     0.00     0.0 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00
19         ts-order-other-mongo    log.row_count           25                            25     1    10     1.40     1.8 2024-01-23 23:06:00+00:00 2024-01-23 23:30:00+00:00, {'total_error_rows': 0, 'error_rows_within_window': 0}, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing dataframes if present; otherwise load and parse timestamps
try:
    metric_df
    trace_df
    log_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_df = pd.read_csv('error_logs.csv')

# Ensure timestamps are parsed as UTC datetimes (safe if already parsed)
for df in (metric_df, trace_df, log_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp('2024-01-23 23:03:29', tz='UTC')
window_end   = pd.Timestamp('2024-01-23 23:33:29', tz='UTC')

candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# Precompute in-window masks for each dataframe
metric_in_window_mask = (metric_df['timestamp'] >= window_start) & (metric_df['timestamp'] <= window_end)
trace_in_window_mask  = (trace_df['timestamp']  >= window_start) & (trace_df['timestamp']  <= window_end)
log_in_window_mask    = (log_df['timestamp']    >= window_start) & (log_df['timestamp']    <= window_end)

rows = []
for c in candidates:
    total_metric_rows = int((metric_df['cmdb_id'] == c).sum())
    metric_rows_within_incident_window = int(((metric_df['cmdb_id'] == c) & metric_in_window_mask).sum())
    total_trace_rows = int((trace_df['cmdb_id'] == c).sum())
    trace_rows_within_incident_window = int(((trace_df['cmdb_id'] == c) & trace_in_window_mask).sum())
    total_log_rows = int((log_df['cmdb_id'] == c).sum())
    log_rows_within_incident_window = int(((log_df['cmdb_id'] == c) & log_in_window_mask).sum())
    rows.append({
        'cmdb_id': c,
        'total_metric_rows': total_metric_rows,
        'metric_rows_within_incident_window': metric_rows_within_incident_window,
        'total_trace_rows': total_trace_rows,
        'trace_rows_within_incident_window': trace_rows_within_incident_window,
        'total_log_rows': total_log_rows,
        'log_rows_within_incident_window': log_rows_within_incident_window
    })

result_df = pd.DataFrame(rows)

# Display the compact result (one row per candidate, up to 5 rows)
result_df

```
Out[2]:
```
Summary of telemetry presence for the five candidate services (incident window 2024-01-23 23:03:29–23:33:29 UTC):

- ts-auth-service: total_metric_rows=175 (175 in window), total_trace_rows=300 (300 in window), total_log_rows=50 (50 in window).
- ts-order-service: total_metric_rows=175 (175 in window), total_trace_rows=900 (900 in window), total_log_rows=50 (50 in window).
- ts-route-service: total_metric_rows=175 (175 in window), total_trace_rows=500 (500 in window), total_log_rows=50 (50 in window).
- ts-train-service: total_metric_rows=175 (175 in window), total_trace_rows=504 (504 in window), total_log_rows=0 (0 in window).
- ts-travel-service: total_metric_rows=200 (200 in window), total_trace_rows=1200 (1200 in window), total_log_rows=50 (50 in window).

Key observations:
- For every candidate, all available telemetry rows fall inside the incident window (total counts equal within-window counts).
- ts-travel-service has the highest trace volume (1200) and the largest metric count (200).
- ts-train-service has no log rows available.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_metric_rows  metric_rows_within_incident_window  total_trace_rows  trace_rows_within_incident_window  total_log_rows  log_rows_within_incident_window
0    ts-auth-service                175                                 175               300                                300              50                               50
1   ts-order-service                175                                 175               900                                900              50                               50
2   ts-route-service                175                                 175               500                                500              50                               50
3   ts-train-service                175                                 175               504                                504               0                                0
4  ts-travel-service                200                                 200              1200                               1200              50                               50```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse metric_df if present; otherwise load and parse timestamps
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidates and incident window (UTC)
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
window_start = pd.Timestamp('2024-01-23 23:03:29', tz='UTC')
window_end   = pd.Timestamp('2024-01-23 23:33:29', tz='UTC')

# Work with full series but restrict to candidate cmdb_ids
metric_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Compute global aggregates per (cmdb_id, kpi_name) on full series (thresholds BEFORE filtering)
grp = metric_cand.groupby(['cmdb_id', 'kpi_name'], dropna=False)
thresholds = grp['value'].quantile([0.05, 0.95]).unstack(level=-1).reset_index().rename(columns={0.05: 'global_p5', 0.95: 'global_p95'})
# Some versions produce column names differently; ensure columns exist
if 'global_p5' not in thresholds.columns or 'global_p95' not in thresholds.columns:
    # If quantile produced differently, reconstruct explicitly
    q05 = grp['value'].quantile(0.05).reset_index().rename(columns={'value': 'global_p5'})
    q95 = grp['value'].quantile(0.95).reset_index().rename(columns={'value': 'global_p95'})
    thresholds = q05.merge(q95, on=['cmdb_id','kpi_name'])

total_counts = grp.size().reset_index(name='total_count')
summary = total_counts.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Now filter to incident window and compute counts and earliest breach timestamp
in_window = metric_cand[(metric_cand['timestamp'] >= window_start) & (metric_cand['timestamp'] <= window_end)].copy()

# count_within_window per series
counts_window = in_window.groupby(['cmdb_id','kpi_name']).agg(count_within_window=('timestamp','size')).reset_index()

# Merge global thresholds into the in-window rows to evaluate breaches
in_window = in_window.merge(summary[['cmdb_id','kpi_name','global_p95','global_p5']], on=['cmdb_id','kpi_name'], how='left')

# Compute breach masks
in_window['above_p95'] = in_window['value'] > in_window['global_p95']
in_window['below_p5'] = in_window['value'] < in_window['global_p5']

# Counts of breaches per series
above_counts = in_window[in_window['above_p95']].groupby(['cmdb_id','kpi_name']).size().reset_index(name='count_within_window_above_p95')
below_counts = in_window[in_window['below_p5']].groupby(['cmdb_id','kpi_name']).size().reset_index(name='count_within_window_below_p5')

# Earliest breach timestamp (either side) per series
in_window['is_breach'] = in_window['above_p95'] | in_window['below_p5']
earliest_breach = in_window[in_window['is_breach']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_timestamp_within_window_of_a_breach'})

# Combine everything
result = summary.merge(counts_window, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(above_counts, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(below_counts, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(earliest_breach, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with zeros
for col in ['count_within_window','count_within_window_above_p95','count_within_window_below_p5']:
    if col in result.columns:
        result[col] = result[col].fillna(0).astype(int)
    else:
        result[col] = 0

# Keep only rows where breaches occurred
result['max_breach_count'] = result[['count_within_window_above_p95','count_within_window_below_p5']].max(axis=1)
result_filtered = result[result['max_breach_count'] > 0].copy()

# Select requested columns and sort
out_cols = ['cmdb_id','kpi_name','total_count','global_p95','global_p5',
            'count_within_window','count_within_window_above_p95','count_within_window_below_p5',
            'earliest_timestamp_within_window_of_a_breach']
# Ensure earliest timestamp column exists
if 'earliest_timestamp_within_window_of_a_breach' not in result_filtered.columns:
    result_filtered['earliest_timestamp_within_window_of_a_breach'] = pd.NaT

result_final = result_filtered.sort_values(by='max_breach_count', ascending=False)[out_cols].reset_index(drop=True).head(20)

# Display compact final result
result_final

```
Out[3]:
```
Summary of metric breaches during 2024-01-23 23:03:29–23:33:29 UTC:

- Which services show metric breaches:
  - ts-auth-service, ts-order-service, and ts-route-service have multiple KPI series that breached their global thresholds (either > global p95 or < global p5) inside the incident window.
  - ts-train-service and ts-travel-service do not appear in the returned breach rows (no series with breaches in the top results).

- What KPIs breached (examples from top rows):
  - ts-auth-service: cpu, diskio, latency-50, latency-90, mem, socket, workload — each series had 25 points in-window and showed 2 values above global p95 and 2 values below global p5.
  - ts-order-service: cpu, diskio, latency-50, latency-90, mem, socket, workload — same pattern: 25 in-window points with 2 above-p95 and 2 below-p5.
  - ts-route-service: cpu, diskio, latency-50, latency-90, mem, socket — same pattern (25 points, 2 above-p95, 2 below-p5).

- Earliest observed breach times (examples):
  - ts-auth-service: earliest breaches vary by KPI (cpu ~2024-01-23 23:12 UTC, diskio ~23:06, latency-50 ~23:18).
  - ts-order-service: cpu ~23:14, diskio ~23:06, latency-50 ~23:13.
  - ts-route-service: cpu/diskio/latency earliest around 23:06–23:11 UTC.

- Interpretation / next steps:
  - Multiple KPIs across auth, order and route services show both spikes above p95 and dips below p5 within the incident window, indicating high variability (transient spikes and drops) rather than a single monotonic shift.
  - Prioritize investigating ts-auth-service, ts-order-service, and ts-route-service (focus on the listed KPIs). Correlate these metric breaches with traces and logs at the breach timestamps to find root cause (e.g., traffic bursts, resource contention, network issues).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_count    global_p95     global_p5  count_within_window  count_within_window_above_p95  count_within_window_below_p5 earliest_timestamp_within_window_of_a_breach
0    ts-auth-service         cpu           25  1.681248e+01  1.379208e+01                   25                              2                             2                    2024-01-23 23:12:00+00:00
1    ts-auth-service      diskio           25  6.419308e+04  4.295088e+04                   25                              2                             2                    2024-01-23 23:06:00+00:00
2    ts-auth-service  latency-50           25  2.847734e-01  2.215556e-01                   25                              2                             2                    2024-01-23 23:18:00+00:00
3    ts-auth-service  latency-90           25  8.679630e-01  4.682106e-01                   25                              2                             2                    2024-01-23 23:12:00+00:00
4    ts-auth-service         mem           25  2.480342e+08  2.445035e+08                   25                              2                             2                    2024-01-23 23:06:00+00:00
5    ts-auth-service      socket           25  1.786000e+01  1.636667e+01                   25                              2                             2                    2024-01-23 23:12:00+00:00
6    ts-auth-service    workload           25  2.895220e+00  2.642620e+00                   25                              2                             2                    2024-01-23 23:16:00+00:00
7   ts-order-service         cpu           25  3.963662e+01  2.123729e+00                   25                              2                             2                    2024-01-23 23:14:00+00:00
8   ts-order-service      diskio           25  9.758222e+04  4.235604e+04                   25                              2                             2                    2024-01-23 23:06:00+00:00
9   ts-order-service  latency-50           25  4.337315e-02  7.603882e-03                   25                              2                             2                    2024-01-23 23:13:00+00:00
10  ts-order-service  latency-90           25  9.705406e-02  2.094788e-02                   25                              2                             2                    2024-01-23 23:12:00+00:00
11  ts-order-service         mem           25  2.807478e+08  2.722318e+08                   25                              2                             2                    2024-01-23 23:06:00+00:00
12  ts-order-service      socket           25  1.518333e+01  1.111000e+01                   25                              2                             2                    2024-01-23 23:09:00+00:00
13  ts-order-service    workload           25  7.452910e+00  4.951507e+00                   25                              2                             2                    2024-01-23 23:09:00+00:00
14  ts-route-service         cpu           25  5.186393e+00  4.472383e+00                   25                              2                             2                    2024-01-23 23:11:00+00:00
15  ts-route-service      diskio           25  6.052048e+04  4.147637e+04                   25                              2                             2                    2024-01-23 23:11:00+00:00
16  ts-route-service  latency-50           25  3.903042e-03  3.589685e-03                   25                              2                             2                    2024-01-23 23:06:00+00:00
17  ts-route-service  latency-90           25  8.598710e-03  7.965220e-03                   25                              2                             2                    2024-01-23 23:06:00+00:00
18  ts-route-service         mem           25  2.573285e+08  2.568523e+08                   25                              2                             2                    2024-01-23 23:07:00+00:00
19  ts-route-service      socket           25  2.192333e+01  2.077602e+01                   25                              2                             2                    2024-01-23 23:06:00+00:00```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse trace_df if present; otherwise load and parse timestamps
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed as UTC
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidates and incident window (UTC)
candidates = ['ts-auth-service','ts-order-service','ts-route-service']
window_start = pd.Timestamp('2024-01-23 23:03:29', tz='UTC')
window_end   = pd.Timestamp('2024-01-23 23:33:29', tz='UTC')

# Restrict to candidates (full series used for global thresholds)
trace_cand = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

# Global aggregates on full series (thresholds BEFORE filtering)
grp = trace_cand.groupby(['cmdb_id','trace_name'], dropna=False)
total_counts = grp.size().reset_index(name='total_count')

p95 = grp['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
p05 = grp['value'].quantile(0.05).reset_index().rename(columns={'value':'global_p5'})

summary = total_counts.merge(p95, on=['cmdb_id','trace_name'], how='left').merge(p05, on=['cmdb_id','trace_name'], how='left')

# Filter to incident window
in_window = trace_cand[(trace_cand['timestamp'] >= window_start) & (trace_cand['timestamp'] <= window_end)].copy()

# Merge global thresholds into in-window rows
in_window = in_window.merge(summary[['cmdb_id','trace_name','global_p95','global_p5']], on=['cmdb_id','trace_name'], how='left')

# Compute breach masks
in_window['above_p95'] = in_window['value'] > in_window['global_p95']
in_window['below_p5'] = in_window['value'] < in_window['global_p5']
in_window['is_breach'] = in_window['above_p95'] | in_window['below_p5']

# Compute counts per (cmdb_id, trace_name)
counts_window = in_window.groupby(['cmdb_id','trace_name']).agg(
    count_within_window = ('timestamp','size'),
    count_within_window_above_p95 = ('above_p95','sum'),
    count_within_window_below_p5 = ('below_p5','sum')
).reset_index()

# Earliest breach timestamp per series
earliest_breach = in_window[in_window['is_breach']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index().rename(
    columns={'timestamp':'earliest_timestamp_within_window_of_a_breach'}
)

# Combine summary (which has total_count and global thresholds) with window counts and earliest breach
result = summary.merge(counts_window, on=['cmdb_id','trace_name'], how='left').merge(
    earliest_breach, on=['cmdb_id','trace_name'], how='left'
)

# Fill NaNs for counts with zeros
for col in ['count_within_window','count_within_window_above_p95','count_within_window_below_p5']:
    result[col] = result[col].fillna(0).astype(int)

# Keep only rows where breaches occurred
result['max_breach_count'] = result[['count_within_window_above_p95','count_within_window_below_p5']].max(axis=1)
result_filtered = result[result['max_breach_count'] > 0].copy()

# Select requested columns and sort by severity descending, limit 20 rows
out_cols = ['cmdb_id','trace_name','total_count','global_p95','global_p5',
            'count_within_window','count_within_window_above_p95','count_within_window_below_p5',
            'earliest_timestamp_within_window_of_a_breach']

# Ensure earliest timestamp column exists
if 'earliest_timestamp_within_window_of_a_breach' not in result_filtered.columns:
    result_filtered['earliest_timestamp_within_window_of_a_breach'] = pd.NaT

trace_breach_final = result_filtered.sort_values(by='max_breach_count', ascending=False)[out_cols].reset_index(drop=True).head(20)

# Display compact final result
trace_breach_final

```
Out[4]:
```
Summary of trace breaches during 2024-01-23 23:03:29–23:33:29 UTC:

- Affected services:
  - ts-auth-service and ts-order-service show multiple trace series with breaches (values > global p95 or < global p5) inside the incident window.
  - ts-route-service does not appear among the returned breach rows.

- ts-auth-service (multiple trace series):
  - Series affected include trace.from_root.duration_mean, trace.from_root.duration_p95, trace.from_root.row_count, trace.from_ts-auth-service.* and trace.to_ts-auth-service.*.
  - Each shown series has 25 points in-window and typically 2 values above the series-wide p95 and 2 below the p5.
  - Earliest breach timestamps vary by series (examples: row_count and some metrics at ~2024-01-23 23:06 UTC; others ~23:08–23:12 UTC).

- ts-order-service (multiple trace series):
  - Affected series include trace.from_root.duration_mean/p95/row_count, trace.from_ts-inside-payment-service.*, trace.from_ts-order-service.*, trace.from_ts-preserve-service.*, etc.
  - Most series show 25 in-window points with ~2 above-p95 and ~2 below-p5; one row_count entry shows 1 above-p95 and 2 below-p5.
  - Earliest breaches appear around ~23:06–23:08 UTC (one row_count earliest at ~23:16 UTC).

- Interpretation and next steps:
  - Traces indicate short-lived spikes and dips (high variability) in auth and order service spans during the incident window rather than a single sustained shift.
  - Prioritize root-cause investigation on ts-auth-service and ts-order-service: inspect span-level details for the breached trace names at the listed earliest-breach times (~23:06–23:12 UTC), and correlate with metrics and logs to identify whether latency spikes, downstream bottlenecks, or traffic bursts caused the anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  total_count  global_p95   global_p5  count_within_window  count_within_window_above_p95  count_within_window_below_p5 earliest_timestamp_within_window_of_a_breach
0    ts-auth-service                      trace.from_root.duration_mean           25    0.373495    0.265032                   25                              2                             2                    2024-01-23 23:12:00+00:00
1    ts-auth-service                       trace.from_root.duration_p95           25    0.985692    0.463232                   25                              2                             2                    2024-01-23 23:08:00+00:00
2    ts-auth-service                          trace.from_root.row_count           25   87.800000   53.000000                   25                              2                             2                    2024-01-23 23:06:00+00:00
3    ts-auth-service           trace.from_ts-auth-service.duration_mean           25    0.003439    0.001994                   25                              2                             2                    2024-01-23 23:12:00+00:00
4    ts-auth-service            trace.from_ts-auth-service.duration_p95           25    0.002830    0.001849                   25                              2                             2                    2024-01-23 23:09:00+00:00
5    ts-auth-service               trace.from_ts-auth-service.row_count           25  350.800000  206.800000                   25                              2                             2                    2024-01-23 23:06:00+00:00
6    ts-auth-service             trace.to_ts-auth-service.duration_mean           25    0.003439    0.001994                   25                              2                             2                    2024-01-23 23:12:00+00:00
7    ts-auth-service              trace.to_ts-auth-service.duration_p95           25    0.002830    0.001849                   25                              2                             2                    2024-01-23 23:09:00+00:00
8    ts-auth-service                 trace.to_ts-auth-service.row_count           25  350.800000  206.800000                   25                              2                             2                    2024-01-23 23:06:00+00:00
9   ts-order-service                      trace.from_root.duration_mean           25    0.344850    0.202962                   25                              2                             2                    2024-01-23 23:06:00+00:00
10  ts-order-service                       trace.from_root.duration_p95           25    0.400456    0.224224                   25                              2                             2                    2024-01-23 23:07:00+00:00
11  ts-order-service                          trace.from_root.row_count           25   13.000000    4.200000                   25                              1                             2                    2024-01-23 23:16:00+00:00
12  ts-order-service  trace.from_ts-inside-payment-service.duration_...           25    0.046945    0.004701                   25                              2                             2                    2024-01-23 23:08:00+00:00
13  ts-order-service  trace.from_ts-inside-payment-service.duration_p95           25    0.080017    0.005883                   25                              2                             2                    2024-01-23 23:12:00+00:00
14  ts-order-service     trace.from_ts-inside-payment-service.row_count           25   12.600000    2.200000                   25                              2                             2                    2024-01-23 23:08:00+00:00
15  ts-order-service          trace.from_ts-order-service.duration_mean           25    0.016125    0.005020                   25                              2                             2                    2024-01-23 23:08:00+00:00
16  ts-order-service           trace.from_ts-order-service.duration_p95           25    0.073949    0.006129                   25                              2                             2                    2024-01-23 23:08:00+00:00
17  ts-order-service              trace.from_ts-order-service.row_count           25  564.600000  253.600000                   25                              2                             2                    2024-01-23 23:06:00+00:00
18  ts-order-service       trace.from_ts-preserve-service.duration_mean           25    0.081169    0.009582                   25                              2                             2                    2024-01-23 23:09:00+00:00
19  ts-order-service        trace.from_ts-preserve-service.duration_p95           25    0.089311    0.010017                   25                              2                             2                    2024-01-23 23:12:00+00:00```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse trace_df if available; otherwise load and parse timestamps
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure timestamp is parsed as UTC
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Substring filters and incident window
substrs = ['to_ts-auth-service','from_ts-auth-service','to_ts-order-service','from_ts-order-service']
pattern = '|'.join(map(repr, substrs)).replace("'", "")  # build simple pattern without extra quotes
# Simpler: use regex join directly
pattern = '|'.join(substrs)

window_start = pd.Timestamp('2024-01-23 23:03:29', tz='UTC')
window_end   = pd.Timestamp('2024-01-23 23:33:29', tz='UTC')

# Filter rows that match any of the substrings in trace_name
mask = trace_df['trace_name'].str.contains(pattern, regex=True, na=False)
trace_filtered = trace_df[mask].copy()

# If no matching rows, return empty DataFrame with requested columns
out_cols = ['cmdb_id','trace_name','total_count','global_p95','global_p5',
            'count_within_window','count_within_window_above_p95','count_within_window_below_p5',
            'earliest_timestamp_within_window_of_a_breach']

if trace_filtered.shape[0] == 0:
    trace_order_auth_breaches = pd.DataFrame(columns=out_cols)
else:
    # Global aggregates on full series (thresholds BEFORE filtering by window)
    grp = trace_filtered.groupby(['cmdb_id','trace_name'], dropna=False)
    total_counts = grp.size().reset_index(name='total_count')
    p95 = grp['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
    p05 = grp['value'].quantile(0.05).reset_index().rename(columns={'value':'global_p5'})
    summary = total_counts.merge(p95, on=['cmdb_id','trace_name'], how='left').merge(p05, on=['cmdb_id','trace_name'], how='left')

    # Rows within incident window
    in_window = trace_filtered[(trace_filtered['timestamp'] >= window_start) & (trace_filtered['timestamp'] <= window_end)].copy()
    # Merge thresholds to evaluate breaches
    in_window = in_window.merge(summary[['cmdb_id','trace_name','global_p95','global_p5']], on=['cmdb_id','trace_name'], how='left')

    in_window['above_p95'] = in_window['value'] > in_window['global_p95']
    in_window['below_p5'] = in_window['value'] < in_window['global_p5']
    in_window['is_breach'] = in_window['above_p95'] | in_window['below_p5']

    # Window-level counts per series
    counts_window = in_window.groupby(['cmdb_id','trace_name']).agg(
        count_within_window = ('timestamp','size'),
        count_within_window_above_p95 = ('above_p95','sum'),
        count_within_window_below_p5 = ('below_p5','sum')
    ).reset_index()

    # Earliest breach timestamp per series
    earliest_breach = in_window[in_window['is_breach']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index().rename(
        columns={'timestamp':'earliest_timestamp_within_window_of_a_breach'}
    )

    # Combine global summary with window counts and earliest breach
    result = summary.merge(counts_window, on=['cmdb_id','trace_name'], how='left').merge(
        earliest_breach, on=['cmdb_id','trace_name'], how='left'
    )

    # Fill NaNs for counts with zeros
    for col in ['count_within_window','count_within_window_above_p95','count_within_window_below_p5']:
        result[col] = result[col].fillna(0).astype(int)

    # Keep only rows with any breaches
    result['max_breach'] = result[['count_within_window_above_p95','count_within_window_below_p5']].max(axis=1)
    result_filtered = result[result['max_breach'] > 0].copy()

    # Select requested columns, sort by count_within_window_above_p95 descending, limit 20
    if 'earliest_timestamp_within_window_of_a_breach' not in result_filtered.columns:
        result_filtered['earliest_timestamp_within_window_of_a_breach'] = pd.NaT

    trace_order_auth_breaches = result_filtered.sort_values(by='count_within_window_above_p95', ascending=False)[out_cols].head(20).reset_index(drop=True)

# Display the compact result
trace_order_auth_breaches

```
Out[5]:
```
Findings for traces between ts-order-service and ts-auth-service (incident window 2024-01-23 23:03:29–23:33:29 UTC):

- Matching trace series were found across multiple components: root-level traces (trace.to_ts-auth-service, trace.to_ts-order-service), ts-auth-service (trace.from/to_ts-auth-service.*), ts-order-service (trace.from/to_ts-order-service.*), and ts-inside-payment-service (trace.to_ts-order-service.*).

- Behavior: each returned series has 25 samples inside the incident window and shows breaches on both sides of its full-series thresholds (typically 2 samples > global p95 and 2 samples < global p5 per series). This indicates short-lived spikes and dips rather than a single sustained shift.

- Timing: earliest breach timestamps are concentrated between about 2024-01-23 23:06:00 UTC and 2024-01-23 23:12:00 UTC across the different series.

- Interpretation: interactions between order and auth (and involvement of an inside-payment service) experienced transient performance variability during the window. This suggests the incident impacted the order↔auth call paths (latency and row-count variability).

- Recommended next steps: inspect full span details and application logs for ts-auth-service, ts-order-service, and ts-inside-payment-service around ~23:06–23:12 UTC; correlate with host/resource metrics and network telemetry to determine whether the cause was service-side latency, downstream contention, or network issues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name  total_count  global_p95   global_p5  count_within_window  count_within_window_above_p95  count_within_window_below_p5 earliest_timestamp_within_window_of_a_breach
0                        root     trace.to_ts-auth-service.duration_mean           25    0.373495    0.265032                   25                              2                             2                    2024-01-23 23:12:00+00:00
1                        root      trace.to_ts-auth-service.duration_p95           25    0.985692    0.463232                   25                              2                             2                    2024-01-23 23:08:00+00:00
2                        root         trace.to_ts-auth-service.row_count           25   87.800000   53.000000                   25                              2                             2                    2024-01-23 23:06:00+00:00
3                        root    trace.to_ts-order-service.duration_mean           25    0.344850    0.202962                   25                              2                             2                    2024-01-23 23:06:00+00:00
4                        root     trace.to_ts-order-service.duration_p95           25    0.400456    0.224224                   25                              2                             2                    2024-01-23 23:07:00+00:00
5             ts-auth-service   trace.from_ts-auth-service.duration_mean           25    0.003439    0.001994                   25                              2                             2                    2024-01-23 23:12:00+00:00
6             ts-auth-service    trace.from_ts-auth-service.duration_p95           25    0.002830    0.001849                   25                              2                             2                    2024-01-23 23:09:00+00:00
7             ts-auth-service       trace.from_ts-auth-service.row_count           25  350.800000  206.800000                   25                              2                             2                    2024-01-23 23:06:00+00:00
8             ts-auth-service      trace.to_ts-auth-service.duration_p95           25    0.002830    0.001849                   25                              2                             2                    2024-01-23 23:09:00+00:00
9             ts-auth-service     trace.to_ts-auth-service.duration_mean           25    0.003439    0.001994                   25                              2                             2                    2024-01-23 23:12:00+00:00
10            ts-auth-service         trace.to_ts-auth-service.row_count           25  350.800000  206.800000                   25                              2                             2                    2024-01-23 23:06:00+00:00
11  ts-inside-payment-service    trace.to_ts-order-service.duration_mean           25    0.046945    0.004701                   25                              2                             2                    2024-01-23 23:08:00+00:00
12           ts-order-service   trace.from_ts-order-service.duration_p95           25    0.073949    0.006129                   25                              2                             2                    2024-01-23 23:08:00+00:00
13  ts-inside-payment-service     trace.to_ts-order-service.duration_p95           25    0.080017    0.005883                   25                              2                             2                    2024-01-23 23:12:00+00:00
14  ts-inside-payment-service        trace.to_ts-order-service.row_count           25   12.600000    2.200000                   25                              2                             2                    2024-01-23 23:08:00+00:00
15           ts-order-service  trace.from_ts-order-service.duration_mean           25    0.016125    0.005020                   25                              2                             2                    2024-01-23 23:08:00+00:00
16           ts-order-service    trace.to_ts-order-service.duration_mean           25    0.016125    0.005020                   25                              2                             2                    2024-01-23 23:08:00+00:00
17           ts-order-service      trace.from_ts-order-service.row_count           25  564.600000  253.600000                   25                              2                             2                    2024-01-23 23:06:00+00:00
18           ts-order-service     trace.to_ts-order-service.duration_p95           25    0.073949    0.006129                   25                              2                             2                    2024-01-23 23:08:00+00:00
19           ts-order-service        trace.to_ts-order-service.row_count           25  564.600000  253.600000                   25                              2                             2                    2024-01-23 23:06:00+00:00```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse metric_df if present; otherwise load and parse timestamps
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed as UTC
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

candidates = ['ts-auth-service','ts-order-service']
window_start = pd.Timestamp('2024-01-23 23:03:29', tz='UTC')
window_end   = pd.Timestamp('2024-01-23 23:33:29', tz='UTC')

# Restrict to candidates but compute global thresholds on their full series (rule 9)
metric_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Global aggregates: total_count and global_p95 per (cmdb_id, kpi_name)
grp = metric_cand.groupby(['cmdb_id','kpi_name'], dropna=False)
total_counts = grp.size().reset_index(name='total_count')
p95 = grp['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})

summary = total_counts.merge(p95, on=['cmdb_id','kpi_name'], how='left')

# Filter to incident window
in_window = metric_cand[(metric_cand['timestamp'] >= window_start) & (metric_cand['timestamp'] <= window_end)].copy()

# Merge global_p95 into in-window rows to evaluate breaches
in_window = in_window.merge(summary[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')

# Compute window-level metrics
# count_within_window, count_within_window_above_p95, max_within_window
in_window['above_p95'] = in_window['value'] > in_window['global_p95']

window_stats = in_window.groupby(['cmdb_id','kpi_name']).agg(
    count_within_window = ('timestamp','size'),
    count_within_window_above_p95 = ('above_p95','sum'),
    max_within_window = ('value','max')
).reset_index()

# Earliest timestamp in-window where value > global_p95
earliest_above = in_window[in_window['above_p95']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(
    columns={'timestamp':'earliest_timestamp_within_window_of_an_above_p95'}
)

# Combine summary with window stats
result = summary.merge(window_stats, on=['cmdb_id','kpi_name'], how='left').merge(
    earliest_above, on=['cmdb_id','kpi_name'], how='left'
)

# Fill NaNs for counts and max where no in-window data
result['count_within_window'] = result['count_within_window'].fillna(0).astype(int)
result['count_within_window_above_p95'] = result['count_within_window_above_p95'].fillna(0).astype(int)
result['max_within_window'] = result['max_within_window'].where(result['max_within_window'].notna(), np.nan)

# Compute severity_ratio = (max_within_window - global_p95)/global_p95 if global_p95>0 else null
def compute_severity(max_val, p95_val):
    if pd.isna(max_val) or pd.isna(p95_val):
        return np.nan
    if p95_val > 0:
        return (max_val - p95_val) / p95_val
    return np.nan

result['severity_ratio'] = result.apply(lambda r: compute_severity(r['max_within_window'], r['global_p95']), axis=1)

# Keep only rows where count_within_window_above_p95 > 0
result_filtered = result[result['count_within_window_above_p95'] > 0].copy()

# Select requested columns and sort by severity_ratio descending, limit 20
out_cols = [
    'cmdb_id','kpi_name','total_count','global_p95',
    'count_within_window','count_within_window_above_p95','max_within_window',
    'earliest_timestamp_within_window_of_an_above_p95','severity_ratio'
]

final = result_filtered.sort_values(by='severity_ratio', ascending=False)[out_cols].reset_index(drop=True).head(20)

# Display compact final result
final

```
Out[6]:
```
Summary of metric breaches (ts-auth-service and ts-order-service) during 2024-01-23 23:03:29–23:33:29 UTC:

Key findings (sorted by severity_ratio):

- ts-order-service · latency-90
  - total_count=25, count_in_window=25, count_above_p95=2
  - earliest above-p95: 2024-01-23 23:20:00 UTC
  - severity_ratio ≈ 0.983 (max ≈ 0.1924 vs p95 ≈ 0.0971) — largest relative spike.

- ts-order-service · diskio
  - total_count=25, count_in_window=25, count_above_p95=2
  - earliest above-p95: 2024-01-23 23:06:00 UTC
  - severity_ratio ≈ 0.392 (max ≈ 135,882 vs p95 ≈ 97,582) — significant IO spike.

- ts-auth-service · diskio
  - total_count=25, count_in_window=25, count_above_p95=2
  - earliest above-p95: 2024-01-23 23:15:00 UTC
  - severity_ratio ≈ 0.203 — notable IO increase on auth host.

- ts-auth-service · latency-50 and several other auth KPIs (workload, cpu, latency-90, socket, mem)
  - each: total_count=25, count_in_window=25, count_above_p95=2
  - earliest above-p95 timestamps range ~23:16–23:24 UTC
  - severity_ratios are smaller (0.03–0.07 range).

- Additional order KPIs with above-p95 events include workload, latency-50, socket, cpu, mem (each with 2 above-p95 samples and small severity_ratio except latency-90 and diskio).

Interpretation and next steps:

- The most severe anomaly by relative magnitude is ts-order-service latency-90 (near doubling of p95) with breaches at ~23:20 UTC — investigate order service latency spikes around that time.
- There are also pronounced disk I/O spikes on ts-order-service (~23:06) and ts-auth-service (~23:15) — investigate host disk activity, I/O saturation, or storage contention.
- Many KPIs show 2 above-p95 samples (and in earlier analyses also some below-p5 samples), indicating short-lived spikes rather than a sustained shift.
- Recommended actions: correlate these breach times (esp. ~23:06, ~23:15, ~23:20) with traces and logs for ts-order-service and ts-auth-service, and inspect host-level I/O, disk latency, and process-level resource usage to determine root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_count    global_p95  count_within_window  count_within_window_above_p95  max_within_window earliest_timestamp_within_window_of_an_above_p95  severity_ratio
0   ts-order-service  latency-90           25  9.705406e-02                   25                              2       1.924388e-01                        2024-01-23 23:20:00+00:00        0.982800
1   ts-order-service      diskio           25  9.758222e+04                   25                              2       1.358829e+05                        2024-01-23 23:06:00+00:00        0.392497
2    ts-auth-service      diskio           25  6.419308e+04                   25                              2       7.721699e+04                        2024-01-23 23:15:00+00:00        0.202887
3    ts-auth-service  latency-50           25  2.847734e-01                   25                              2       3.033655e-01                        2024-01-23 23:24:00+00:00        0.065287
4   ts-order-service    workload           25  7.452910e+00                   25                              2       7.794600e+00                        2024-01-23 23:11:00+00:00        0.045847
5    ts-auth-service    workload           25  2.895220e+00                   25                              2       3.006900e+00                        2024-01-23 23:16:00+00:00        0.038574
6    ts-auth-service         cpu           25  1.681248e+01                   25                              2       1.741959e+01                        2024-01-23 23:24:00+00:00        0.036110
7   ts-order-service  latency-50           25  4.337315e-02                   25                              2       4.476538e-02                        2024-01-23 23:25:00+00:00        0.032099
8    ts-auth-service  latency-90           25  8.679630e-01                   25                              2       8.899859e-01                        2024-01-23 23:16:00+00:00        0.025373
9    ts-auth-service      socket           25  1.786000e+01                   25                              2       1.826667e+01                        2024-01-23 23:16:00+00:00        0.022770
10  ts-order-service      socket           25  1.518333e+01                   25                              2       1.540000e+01                        2024-01-23 23:25:00+00:00        0.014270
11   ts-auth-service         mem           25  2.480342e+08                   25                              2       2.495252e+08                        2024-01-23 23:23:00+00:00        0.006011
12  ts-order-service         cpu           25  3.963662e+01                   25                              2       3.968895e+01                        2024-01-23 23:22:00+00:00        0.001320
13  ts-order-service         mem           25  2.807478e+08                   25                              2       2.807826e+08                        2024-01-23 23:29:00+00:00        0.000124```
```